In [29]:
%load_ext autoreload
%autoreload 2

from dateutil import parser
from zoneinfo import ZoneInfo
from datetime import timezone, timedelta

from pynims.workflows import download_images_for_camera
from pynims.client import NIMSClient
from pynims.utils import convert_nims_image_name_to_utc_date

import dataretrieval.nwis as nwis

import pandas as pd
from IPython.display import display

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
# Get current run config files
import glob
import os 

config_paths = sorted(glob.glob('cameras/*/*/run_config.json'))

for i, cfg in enumerate(config_paths):
    parts = cfg.split(os.sep)
    print(f"Site Select {i}:")
    print(f"camera_id = '{parts[1]}'")
    print(f"run_name  = '{parts[2]}'")
    print()

Site Select 0:
camera_id = 'CA_Arroyo_DE_LA_Laguna_A_Corte_Madrid_nr_Pleasanton'
run_name  = 'event_2026-02-15'

Site Select 1:
camera_id = 'OK_Illinois_River_near_Moodys'
run_name  = 'event_2026-03-04'

Site Select 2:
camera_id = 'SC_Waccamaw_River_at_SC_22_below_Longs'
run_name  = 'event_2025-08-14'

Site Select 3:
camera_id = 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX'
run_name  = 'event_2026-03-16'

Site Select 4:
camera_id = 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet'
run_name  = 'event_2025-06-23'

Site Select 5:
camera_id = 'WI_East_Branch_Pecatonica_River_near_Blanchardville_Bullet'
run_name  = 'test_2025-03-25'



In [53]:
import json, os
# Select site from above to load existing run config if available (for reproducibility)
site_select = 3


########################################
_config_path = config_paths[site_select]
_existing_config = {}
if os.path.exists(_config_path):
    with open(_config_path) as f:
        _existing_config = json.load(f)
    print(f'Loaded existing config from {_config_path}')

    site = _existing_config.get('site')
    camera_id = _existing_config.get('camera_id')
    run_name = _existing_config.get('run_name')
    max_results = _existing_config.get('max_results')

    run_dir = f'cameras/{camera_id}/{run_name}'
    save_dir = f'{run_dir}/images'

    start_all = _existing_config.get('start_all')
    end_all = _existing_config.get('end_all')

    start_event = _existing_config.get('start_event')
    end_event = _existing_config.get('end_event')

    use_event_times = _existing_config.get('use_event_times')
    
    allowable_im_data_time_diff = _existing_config.get('allowable_im_data_time_diff')



Loaded existing config from cameras/VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX/event_2026-03-16/run_config.json


In [54]:
with NIMSClient() as client:
    tz = client.get_camera_attribute(camera_id, "tz")
print(f"==>> tz: {tz}")

if use_event_times:
    start = parser.parse(start_event)
    end = parser.parse(end_event)
else:
    start = parser.parse(start_all)
    end = parser.parse(end_all)


# Attach known timezone (if not already present)
if start.tzinfo is None:
    start = start.replace(tzinfo=ZoneInfo(tz))
if end.tzinfo is None:
    end = end.replace(tzinfo=ZoneInfo(tz))

# Convert to UTC
start = start.astimezone(timezone.utc)
print(f"==>> start (utc): {start}")
end = end.astimezone(timezone.utc)
print(f"==>> end (utc): {end}")

==>> tz: US/Eastern
==>> start (utc): 2026-03-16 12:00:00+00:00
==>> end (utc): 2026-03-17 10:00:00+00:00


In [55]:
with NIMSClient() as client:
    image_list = client.get_image_list(
        camera_id, start, end, recursive=None, max_results=max_results
    )
print(f"==>> image_list: {image_list}")
print(f"==>> len(image_list): {len(image_list)}")

==>> image_list: ['VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T12-00-02Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T12-15-04Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T12-30-04Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T12-45-01Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T13-00-03Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T13-15-03Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T13-30-04Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T13-45-02Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T14-00-03Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T14-15-05Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T14-30-02Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T14-45-02Z.jpg', 'VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX___2026-03-16T15-00-03Z.jpg', 'VA_DIF

In [56]:
keep_every_nth_image = _existing_config.get('keep_every_nth_image', 1) # set to 1 to keep all images
image_list = image_list[0::keep_every_nth_image]
print(f"==>> len(image_list) after keeping every {keep_every_nth_image}th image: {len(image_list)}")

==>> len(image_list) after keeping every 1th image: 88


In [57]:
download_images_for_camera(camera_id, start, end, max_results=max_results, save_dir=save_dir, image_list=image_list)

image # 1 of 88
image # 2 of 88
image # 3 of 88
image # 4 of 88
image # 5 of 88
image # 6 of 88
image # 7 of 88
image # 8 of 88
image # 9 of 88
image # 10 of 88
image # 11 of 88
image # 12 of 88
image # 13 of 88
image # 14 of 88
image # 15 of 88
image # 16 of 88
image # 17 of 88
image # 18 of 88
image # 19 of 88
image # 20 of 88
image # 21 of 88
image # 22 of 88
image # 23 of 88
image # 24 of 88
image # 25 of 88
image # 26 of 88
image # 27 of 88
image # 28 of 88
image # 29 of 88
image # 30 of 88
image # 31 of 88
image # 32 of 88
image # 33 of 88
image # 34 of 88
image # 35 of 88
image # 36 of 88
image # 37 of 88
image # 38 of 88
image # 39 of 88
image # 40 of 88
image # 41 of 88
image # 42 of 88
image # 43 of 88
image # 44 of 88
image # 45 of 88
image # 46 of 88
image # 47 of 88
image # 48 of 88
image # 49 of 88
image # 50 of 88
image # 51 of 88
image # 52 of 88
image # 53 of 88
image # 54 of 88
image # 55 of 88
image # 56 of 88
image # 57 of 88
image # 58 of 88
image # 59 of 88
image 

In [58]:
image_times = [convert_nims_image_name_to_utc_date(image) for image in image_list]
print([dt.isoformat() for dt in image_times])
print(f"==>> len(image_times): {len(image_times)}")

['2026-03-16T12:00:02+00:00', '2026-03-16T12:15:04+00:00', '2026-03-16T12:30:04+00:00', '2026-03-16T12:45:01+00:00', '2026-03-16T13:00:03+00:00', '2026-03-16T13:15:03+00:00', '2026-03-16T13:30:04+00:00', '2026-03-16T13:45:02+00:00', '2026-03-16T14:00:03+00:00', '2026-03-16T14:15:05+00:00', '2026-03-16T14:30:02+00:00', '2026-03-16T14:45:02+00:00', '2026-03-16T15:00:03+00:00', '2026-03-16T15:15:03+00:00', '2026-03-16T15:30:02+00:00', '2026-03-16T15:45:03+00:00', '2026-03-16T16:00:03+00:00', '2026-03-16T16:15:02+00:00', '2026-03-16T16:30:02+00:00', '2026-03-16T16:45:02+00:00', '2026-03-16T17:00:02+00:00', '2026-03-16T17:15:22+00:00', '2026-03-16T17:30:02+00:00', '2026-03-16T17:45:02+00:00', '2026-03-16T18:00:02+00:00', '2026-03-16T18:15:05+00:00', '2026-03-16T18:30:02+00:00', '2026-03-16T18:45:05+00:00', '2026-03-16T19:00:03+00:00', '2026-03-16T19:15:02+00:00', '2026-03-16T19:30:02+00:00', '2026-03-16T19:45:04+00:00', '2026-03-16T20:00:02+00:00', '2026-03-16T20:15:02+00:00', '2026-03-16T2

In [59]:
# Data request requires start and end date in format YYYY-MM-DD
start_minus_one_day = (start - timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0).strftime("%Y-%m-%d")
end_plus_one_day = (end + timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0).strftime("%Y-%m-%d")

# Get data
data_df = nwis.get_record(sites=site, service='iv', start=start_minus_one_day, end=end_plus_one_day)

# Filter data to our original range
mask = (data_df.index >= start) & (data_df.index <= end)
data_df = data_df.loc[mask]

display(data_df)

,site_no,00010,00010_cd,00060,00060_cd,00065,00065_cd,00095,00095_cd,00300,00300_cd,00400,00400_cd,63680_ysi exo,63680_ysi exo_cd,99234,99234_cd
datetime,,,,,,,,,,,,,,,,,
2026-03-16 12:00:00+00:00,01645704,10.9,P,6.19,P,1.12,P,1120.0,P,9.7,P,7.4,P,4.7,P,2.0,P
2026-03-16 12:05:00+00:00,01645704,NaN,NaN,6.19,P,1.12,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-16 12:10:00+00:00,01645704,NaN,NaN,6.19,P,1.12,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-16 12:15:00+00:00,01645704,11.0,P,6.19,P,1.12,P,1100.0,P,9.7,P,7.4,P,4.6,P,2.0,P
2026-03-16 12:20:00+00:00,01645704,NaN,NaN,6.19,P,1.12,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-17 09:40:00+00:00,01645704,NaN,NaN,47.70,P,1.88,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-03-17 09:45:00+00:00,01645704,9.8,P,47.70,P,1.88,P,357.0,P,10.4,P,7.1,P,52.9,P,12.0,P
2026-03-17 09:50:00+00:00,01645704,NaN,NaN,47.10,P,1.87,P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
image_df = pd.DataFrame({"image_times": image_times, "image_names": image_list})
display(image_df)

,image_times,image_names
0,2026-03-16 12:00:02+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...
1,2026-03-16 12:15:04+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...
2,2026-03-16 12:30:04+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...
3,2026-03-16 12:45:01+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...
4,2026-03-16 13:00:03+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...
...,...,...
83,2026-03-17 08:45:22+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...
84,2026-03-17 09:00:02+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...
85,2026-03-17 09:15:02+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...
86,2026-03-17 09:30:02+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...


In [61]:
merged = pd.merge_asof(
    image_df.sort_values("image_times"),
    data_df.reset_index().rename(columns={"datetime": "data_times"}),
    left_on="image_times",
    right_on="data_times",
    direction="nearest",
)

merged["time_diff_sec"] = (merged["image_times"] - merged["data_times"]).abs().dt.total_seconds()
filtered = merged[merged["time_diff_sec"] <= allowable_im_data_time_diff]

display(merged)

,image_times,image_names,data_times,site_no,00010,00010_cd,00060,00060_cd,00065,00065_cd,...,00095_cd,00300,00300_cd,00400,00400_cd,63680_ysi exo,63680_ysi exo_cd,99234,99234_cd,time_diff_sec
0,2026-03-16 12:00:02+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...,2026-03-16 12:00:00+00:00,01645704,10.9,P,6.19,P,1.12,P,...,P,9.7,P,7.4,P,4.7,P,2.0,P,2.0
1,2026-03-16 12:15:04+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...,2026-03-16 12:15:00+00:00,01645704,11.0,P,6.19,P,1.12,P,...,P,9.7,P,7.4,P,4.6,P,2.0,P,4.0
2,2026-03-16 12:30:04+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...,2026-03-16 12:30:00+00:00,01645704,11.0,P,6.19,P,1.12,P,...,P,9.7,P,7.4,P,4.7,P,2.0,P,4.0
3,2026-03-16 12:45:01+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...,2026-03-16 12:45:00+00:00,01645704,11.1,P,5.90,P,1.11,P,...,P,9.8,P,7.4,P,4.6,P,2.0,P,1.0
4,2026-03-16 13:00:03+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...,2026-03-16 13:00:00+00:00,01645704,11.1,P,5.90,P,1.11,P,...,P,9.8,P,7.4,P,4.6,P,2.0,P,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,2026-03-17 08:45:22+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...,2026-03-17 08:45:00+00:00,01645704,10.1,P,52.40,P,1.95,P,...,P,10.4,P,7.1,P,57.7,P,12.0,P,22.0
84,2026-03-17 09:00:02+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...,2026-03-17 09:00:00+00:00,01645704,10.0,P,51.10,P,1.93,P,...,P,10.4,P,7.1,P,56.5,P,12.0,P,2.0
85,2026-03-17 09:15:02+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...,2026-03-17 09:15:00+00:00,01645704,9.9,P,49.70,P,1.91,P,...,P,10.4,P,7.1,P,59.6,P,12.0,P,2.0
86,2026-03-17 09:30:02+00:00,VA_DIFFICULT_RUN_ABOVE_FOX_LAKE_NEAR_FAIRFAX__...,2026-03-17 09:30:00+00:00,01645704,9.8,P,48.40,P,1.89,P,...,P,10.4,P,7.1,P,54.4,P,12.0,P,2.0


In [ ]:
# merged.to_csv(f'{run_dir}/images_and_data.csv')

In [ ]:
# run_config = {
#     "camera_id": camera_id,
#     "site": site,
#     "run_name": run_name,
#     "start_all": start_all,
#     "end_all": end_all,
#     "start_event": start_event,
#     "end_event": end_event,
#     "use_event_times": use_event_times,
#     "max_results": max_results,
#     "keep_every_nth_image": keep_every_nth_image,
#     "allowable_im_data_time_diff": allowable_im_data_time_diff,
# }

# os.makedirs(run_dir, exist_ok=True)
# config_path = f'{run_dir}/run_config.json'
# with open(config_path, 'w') as f:
#     json.dump(run_config, f, indent=2)

# print(f"Run config saved to {config_path}")